# GANg quick start

Train GANg on 1,000 real QMOF MOFs, generate descriptor-space MOFs near a requested band gap, attach them to the real similarity graph and audit them.

`pip install "gang-mof @ git+https://github.com/MehrdadJalali-AI/GANg"` — runs on a laptop CPU in about 2–3 minutes.

In [1]:
from gang_mof import GANg, load_qmof_example

data = load_qmof_example()
train, validation, test = data.split(random_state=42)
data, len(train), len(validation), len(test)

(NodeDataset(n_nodes=1000, n_features=106, schema='qmof106', target='bandgap'),
 700,
 150,
 150)

## 1. Train
Validation nodes are used only to select the checkpoint (coverage of validation descriptors).

In [2]:
model = GANg(random_state=0)
model.fit(train, validation_data=validation)

[GANg] training constrained on 700 nodes x 106 descriptors (3000 generator updates, device=cpu)


[GANg] update  1000/3000  validation coverage 0.009  (43s)


[GANg] update  1250/3000  validation coverage 0.049  (54s)


[GANg] update  1500/3000  validation coverage 0.147  (65s)


[GANg] update  1750/3000  validation coverage 0.224  (76s)


...

[GANg] update  2500/3000  validation coverage 0.620  (107s)


[GANg] update  2750/3000  validation coverage 0.716  (118s)


[GANg] update  3000/3000  validation coverage 0.724  (129s)


[GANg] selected checkpoint at update 3000 (validation coverage 0.724)


GANg(fitted, constrained, conditional=True, iterations=3000)

## 2. Generate and attach
The requested band gap is stored as `requested_condition`; it is *not* a label. Edges are cosine kNN links to real training MOFs.

In [3]:
result = model.sample(n_samples=200, condition=2.0)
model.attach(result, k=15)
result.nodes[['node_id', 'formula', 'requested_condition', 'info_lcd', 'info_density']].head()

,node_id,formula,requested_condition,info_lcd,info_density
0,gen-000001,H9C8N10O7V2Zn1Mo2,2.0,2.984770,2.443842
1,gen-000002,H9C11N2S2Cl2,2.0,2.312527,1.667936
2,gen-000003,H17C15N6O6Co1Zn1,2.0,3.340700,1.660136
3,gen-000004,H30C24O5,2.0,2.432958,1.138410
4,gen-000005,H6C7N1O5,2.0,1.984074,1.539123


## 3. Audit against held-out MOFs

In [4]:
metrics = model.evaluate(result, reference=test)
print(result.summary())

GANg generation: 200 nodes (schema: qmof106, mode: constrained)
  accepted:               0.740
  within training range:  0.740
  schema consistent:      1.000
  novel (vs training):    0.995
  duplicate of training:  0.000
  edges attached:         3000 (k = 15)
  coverage:               0.740
  precision:              0.830
  recall:                 0.713
  pairwise_distance_ratio:1.096
  top1_analog_share:      0.075


In [5]:
result.checks.head()

,node_id,finite,n_features_out_of_range,within_training_range,schema_consistent,nearest_training_similarity,novelty_score,duplicate_of_training,duplicate_in_batch,novel,rejection_reasons,accepted
0,gen-000001,True,0,True,True,0.764681,0.235319,False,False,True,,True
1,gen-000002,True,1,False,True,0.795531,0.204469,False,False,True,within_training_range,False
2,gen-000003,True,0,True,True,0.735374,0.264626,False,False,True,,True
3,gen-000004,True,2,False,True,0.895832,0.104168,False,False,True,within_training_range,False
4,gen-000005,True,2,False,True,0.947628,0.052372,False,False,True,within_training_range,False


Nearest real QMOF analogs (interpretation only):

In [6]:
result.analogs.head(10)

,node_id,rank,analog_id,similarity,analog_target,analog_formula
0,gen-000001,1,qmof-0505344,0.764681,1.292418,Ag4Mo4C10H6N12O13
1,gen-000001,2,qmof-06933eb,0.756196,2.385175,V2ZnC12H8N3O6
2,gen-000001,3,qmof-c9bb6d6,0.752309,1.732814,Ag4Mo3C12H8N10O10
3,gen-000001,4,qmof-cea07ef,0.736596,1.853504,CuMo2C11H7N6O8
4,gen-000001,5,qmof-74c5a19,0.659410,1.901844,Co2V8C48H32N8O23
5,gen-000002,1,qmof-ea2e5ae,0.795531,2.454198,UC11H17NO7
6,gen-000002,2,qmof-d35f64f,0.771508,2.105572,UC12H8N2O8
7,gen-000002,3,qmof-cd5c084,0.751770,3.483867,TlC7H5N4
8,gen-000002,4,qmof-4df225e,0.743499,2.507743,UC6H8O6
9,gen-000002,5,qmof-9dedbbd,0.698268,3.175296,TlBrC7H4N4


## 4. Export and save

In [7]:
result.export('outputs/notebook_results')
model.save('outputs/notebook_model')
reloaded = GANg.load('outputs/notebook_model', verbose=False)
reloaded.sample(3, condition=1.0).nodes[['node_id', 'formula', 'requested_condition']]

,node_id,formula,requested_condition
0,gen-000001,H10C9N11O7V3Zn1Mo2,1.0
1,gen-000002,H9C12N2S2Cl2,1.0
2,gen-000003,H19C17N8O6Co1Zn1,1.0
